# Redrob Hackathon — Intelligent Candidate Ranking (Sandbox Demo)

**Team:** CampusCollab

This notebook runs our candidate-ranking engine **end-to-end on CPU** and produces a ranked CSV — exactly as it runs for the full 100K submission, just on a small sample.

### What this engine is
A **glass-box, fully deterministic, zero-AI, zero-network** ranker. No LLM calls, no embeddings, no GPU, no model downloads. Pure feature-engineered scoring + classical statistical ML (population calibration, Isolation Forest, bootstrap confidence). Every score is an auditable decomposition, and the reasoning column is a deterministic readout of that same score — so it can never contradict the rank or hallucinate.

### How to use
**Just press `Runtime → Run all`.** It will:
1. Install dependencies (numpy, scikit-learn)
2. Load a built-in 12-candidate demo set (or your uploaded file)
3. Rank it and display the results
4. Let you download the CSV

The demo set is curated to show the engine handling every trap type the JD describes — strong fits rise, keyword-stuffers and honeypots are floored, plain-language fits are caught, and unresponsive candidates are down-weighted.

## 1. Install dependencies

In [ ]:
!pip install numpy scikit-learn -q
print('dependencies ready')

## 2. Load the ranking engine
This writes the full engine (`rank.py`) to disk, identical to the one in our GitHub repo.

In [ ]:
%%writefile rank.py
#!/usr/bin/env python3
"""
rank.py — Redrob Intelligent Candidate Discovery & Ranking Engine v2
=====================================================================
Glass-box, deterministic, zero-network, CPU-only candidate ranker.
Now with: multiprocessing across all cores, population-calibrated scoring,
Isolation Forest anomaly detection, and bootstrap rank-confidence.

ARCHITECTURE (five phases after loading)
-----------------------------------------
  Phase 1  Extract raw features — multiprocessing across all CPU cores
  Phase 2  Population calibration (Concept A) — percentile-normalize the
           most arbitrary pillars against the real 100K distribution
  Phase 3  Isolation Forest (Concept B) — unsupervised second-opinion on
           honeypots; catches patterns no hand-written rule anticipated
  Phase 4  Bootstrap rank-confidence (Concept C) — 500 weight perturbations
           via matrix multiply; confidence interval per candidate
  Phase 5  Final rank, trace-driven reasoning (now includes confidence),
           spec-compliant CSV

SCORING LAYERS (within each candidate's feature extraction)
------------------------------------------------------------
  L0  Honeypot / impossible-profile gate         → hard floor
  L1  Role-fit gate                              → floors keyword-stuffers
  L2  Domain evidence (layered explicit/adjacent)→ reads career descriptions
  L3  Fit pillars                                → 9 weighted components
  L4  Negative do-NOT-want penalties             → multiplicative deduction
  L5  Behavioral availability multiplier         → JD "actually available"

Usage
-----
    python rank.py --candidates candidates.jsonl.gz --out submission.csv
    python rank.py --candidates sample_candidates.json --out sample_out.csv --top 25
"""

import argparse
import csv
import gzip
import json
import math
import os
import re
from datetime import date, datetime
from multiprocessing import Pool, cpu_count

import numpy as np

try:
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

# ──────────────────────────────────────────────────────────────────────────────
# JOB PROFILE — hand-derived from the REAL Redrob JD
# ──────────────────────────────────────────────────────────────────────────────
JOB = {
    "title": "Senior AI Engineer — Founding Team (Redrob)",
    "ideal_yoe": 7.0, "yoe_sigma": 2.0,
    "yoe_soft_min": 4.0,
    "target_cities": {
        "pune", "noida", "hyderabad", "mumbai", "delhi", "new delhi",
        "ncr", "gurgaon", "gurugram", "ghaziabad", "faridabad", "greater noida",
    },
    "good_india_cities": {"bangalore", "bengaluru", "chennai", "kolkata", "ahmedabad"},
    "notice_ideal_days": 30,
}

WEIGHTS = {
    "domain_evidence":     0.20,  # 0.20 with semantic; 0.30 via fallback when precomputed embeddings absent
    "skill_substance":     0.15,
    "seniority_fit":       0.12,
    "product_vs_services": 0.10,
    "external_validation": 0.08,
    "eval_frameworks":     0.08,
    "semantic_similarity": 0.10,  # NEW: cosine sim against JD embedding (msmarco-distilbert-cos-v5)
    "python_signal":       0.02,
    "location":            0.06,
    "notice":              0.04,
    "platform_quality":    0.05,
}
WEIGHTS_FALLBACK = {
    "domain_evidence":     0.30,
    "skill_substance":     0.15,
    "seniority_fit":       0.12,
    "product_vs_services": 0.10,
    "external_validation": 0.08,
    "eval_frameworks":     0.08,
    "python_signal":       0.02,
    "location":            0.06,
    "notice":              0.04,
    "platform_quality":    0.05,
}
PILLAR_KEYS = list(WEIGHTS.keys())

# Pillars where absolute values are arbitrary → population-calibrate these
CALIBRATE_PILLARS = {"domain_evidence", "skill_substance",
                     "eval_frameworks", "semantic_similarity"}

# Bootstrap settings
N_BOOTSTRAP = 500
BOOTSTRAP_SEED = 42

# ──────────────────────────────────────────────────────────────────────────────
# LEXICONS  (case-sensitive; all inputs are pre-lowercased)
# ──────────────────────────────────────────────────────────────────────────────
def _rx(words):
    return re.compile("|".join(r"\b" + w + r"\b" for w in words))

RX_RETRIEVAL = _rx([
    r"retriev\w*", r"ranking", r"rank(ed|ing|er)?", r"recommend\w*", r"recommender",
    r"search relevance", r"relevance", r"semantic search", r"vector search",
    r"nearest neighbou?r", r"learning[- ]to[- ]rank", r"\bltr\b", r"\bapproximate\s+nearest\b",
    r"information retrieval", r"information\s+retrieval", r"personali[sz]ation", r"matching",
    r"\bbm25\b", r"\bfaiss\b", r"elasticsearch", r"opensearch", r"\bsolr\b",
    r"\blucene\b", r"embeddings?", r"two[- ]tower", r"candidate generation",
    r"collaborative\s+filter\w*", r"matrix\s+factori\w*", r"item.item\s+similar\w*",
    r"session.based\s+recommend\w*", r"cold\s+start",
    r"document\s+(?:ranking|retrieval|scoring)",
    r"passage\s+(?:retrieval|ranking|reranking)",
    r"query\s+(?:expansion|understanding|rewriting|relevance)",
])
RX_EVAL = _rx([
    r"ndcg", r"\bmrr\b", r"mean reciprocal", r"mean average precision", r"\bmap@?\d*\b",
    r"a/?b test\w*", r"offline eval\w*", r"online eval\w*", r"precision@\d+",
    r"recall@\d+", r"\bauc\b", r"holdout", r"counterfactual", r"interleav\w*",
])
RX_LLM_MODERN = _rx([
    r"\bllm\b", r"large language model", r"\brag\b", r"retrieval[- ]augmented",
    r"fine[- ]tun\w*", r"\blora\b", r"\bqlora\b", r"\bpeft\b", r"transformer\w*",
    r"\bbert\b", r"sentence[- ]transformer", r"\be5\b", r"\bbge\b", r"reranke?\w*",
])
RX_VECTOR_DB = _rx([
    r"pinecone", r"weaviate", r"qdrant", r"milvus", r"\bfaiss\b", r"pgvector",
    r"elasticsearch", r"opensearch", r"vector (db|database|store|index)",
])
RX_PRODUCTION = _rx([
    r"production", r"deployed?", r"at scale", r"real users", r"serving",
    r"latency", r"throughput", r"\bpipeline\b", r"in[- ]production", r"shipped",
    r"\bmlops\b", r"\bapi\b", r"microservice\w*",
    r"click.through\s+rate", r"\bctr\b", r"conversion\s+rate",
    r"e.commerce\s+(?:search|ranking|recommendation|relevance)",
])
RX_CV_SPEECH_ROBO = _rx([
    r"computer vision", r"image classification", r"object detection",
    r"segmentation", r"\bcv\b", r"speech recognition", r"\basr\b",
    r"text[- ]to[- ]speech", r"\btts\b", r"robotics", r"\bslam\b", r"lidar",
    r"point cloud", r"autonomous (driving|vehicle)",
])
RX_NLP_IR = _rx([
    r"\bnlp\b", r"natural language", r"text mining", r"named entity",
    r"\bner\b", r"question answering", r"summari[sz]ation", r"information retrieval",
])
RX_ADJACENT_ML = _rx([
    r"feature (pipeline|engineering|store)", r"feature pipelines?", r"ml pipeline\w*",
    r"experimentation", r"model (training|serving|deployment|inference)",
    r"data science", r"machine learning model\w*", r"predictive model\w*",
    r"classification", r"regression model\w*", r"forecasting", r"churn", r"propensity",
    r"\bkaggle\b", r"fine[- ]tun\w*", r"\bspark\b", r"airflow", r"\bdbt\b", r"\betl\b",
    r"\bxgboost\b", r"lightgbm", r"gradient boost\w*",
    r"data (pipeline|infrastructure|warehouse)", r"analytics", r"\bml\b model\w*",
])
RX_PYTHON   = _rx([r"python", r"pytorch", r"tensorflow", r"scikit", r"numpy", r"pandas"])
RX_OSS      = _rx([r"open[- ]source", r"github", r"maintainer", r"contributor",
                   r"\bpaper\b", r"published", r"\btalk\b", r"conference"])
RX_RESEARCH = _rx([r"research scientist", r"\bphd\b", r"post[- ]?doc", r"postdoctoral",
                   r"research (lab|assistant|associate)", r"academ\w*", r"thesis",
                   r"university research", r"published \d"])

RX_NON_TECHNICAL = _rx([
    r"hr\b", r"human resource\w*", r"recruit\w*", r"talent acquisition",
    r"marketing", r"content writer", r"copywriter", r"\bcontent\b", r"\bsales\b",
    r"account executive", r"accountant", r"\bfinance\b", r"customer support",
    r"customer success", r"operations manager", r"\bbpo\b", r"administrativ\w*",
    r"office manager", r"business development", r"social media",
])
RX_NON_TARGET_TECH = _rx([
    r"civil engineer", r"mechanical engineer", r"electrical engineer",
    r"hardware", r"network engineer", r"\bvlsi\b", r"chemical engineer",
])
RX_CORE_ML = _rx([
    r"machine learning", r"\bml\b", r"\bai\b engineer", r"\bai\b", r"applied scientist",
    r"data scientist", r"research engineer", r"\bnlp\b", r"search", r"ranking",
    r"relevance", r"deep learning", r"mlops", r"ml engineer", r"ai engineer",
    r"research scientist", r"ml scientist",
])
RX_ADJACENT_ENG = _rx([
    r"software engineer", r"backend engineer", r"back[- ]end", r"data engineer",
    r"\bsde\b", r"full[- ]?stack", r"platform engineer", r"developer", r"programmer",
    r"devops", r"software developer", r"engineer",
])

SERVICES_FIRMS = {
    "tcs", "tata consultancy", "infosys", "wipro", "accenture", "cognizant",
    "capgemini", "tech mahindra", "hcl", "hcltech", "mindtree", "ltimindtree",
    "lti", "deloitte", "ibm", "dxc", "mphasis", "persistent systems",
}
PROF_W = {"beginner": 0.3, "intermediate": 0.6, "advanced": 0.85, "expert": 1.0}

# ──────────────────────────────────────────────────────────────────────────────
# DATA LOADING
# ──────────────────────────────────────────────────────────────────────────────
def load_candidates(path):
    out = []
    if path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    out.append(json.loads(line))
    elif path.endswith(".jsonl"):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    out.append(json.loads(line))
    else:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            out = data if isinstance(data, list) else [data]
    return out

def _parse_date(s):
    if not s or len(s) < 10:
        return None
    try:
        return date(int(s[:4]), int(s[5:7]), int(s[8:10]))
    except Exception:
        return None

def _candidate_text(c):
    p = c.get("profile", {})
    parts = [p.get("headline", ""), p.get("summary", ""),
             p.get("current_title", ""), p.get("current_industry", "")]
    for h in c.get("career_history", []):
        parts.append(h.get("title", ""))
        parts.append(h.get("description", ""))
        parts.append(h.get("industry", ""))
    for s in c.get("skills", []):
        parts.append(s.get("name", ""))
    return " \n ".join(x for x in parts if x)

# ──────────────────────────────────────────────────────────────────────────────
# L0 — HONEYPOT DETECTION
# ──────────────────────────────────────────────────────────────────────────────
def honeypot_flags(c):
    flags = []
    yoe = float(c.get("profile", {}).get("years_of_experience") or 0)
    total_career_m = sum(int(h.get("duration_months") or 0) for h in c.get("career_history", []))
    zero_dur_high = sum(
        1 for s in c.get("skills", [])
        if s.get("proficiency") in ("advanced", "expert")
        and int(s.get("duration_months") or 0) == 0
    )
    if zero_dur_high >= 3:
        flags.append("many_high_skills_zero_duration")
    for s in c.get("skills", []):
        career_baseline = max(total_career_m, int(yoe * 12))
        if int(s.get("duration_months") or 0) > career_baseline + 12 and career_baseline > 0:
            flags.append("skill_duration_exceeds_career"); break
    date_conflicts = 0
    for h in c.get("career_history", []):
        sd = _parse_date(h.get("start_date"))
        ed = _parse_date(h.get("end_date")) or date(2026, 6, 1)
        dur = int(h.get("duration_months") or 0)
        if sd:
            calc = (ed.year - sd.year) * 12 + (ed.month - sd.month)
            if calc >= 0 and abs(dur - calc) > 9:
                date_conflicts += 1
    if date_conflicts >= 1:
        flags.append("tenure_date_contradiction")
    if yoe > 0 and total_career_m > yoe * 12 + 30:
        flags.append("career_sum_exceeds_yoe")
    currents = [h for h in c.get("career_history", []) if h.get("is_current")]
    if any(h.get("end_date") for h in currents) or len(currents) > 1:
        flags.append("is_current_contradiction")
    strong = {"many_high_skills_zero_duration", "skill_duration_exceeds_career"}
    is_hp = bool(strong & set(flags)) or len(set(flags)) >= 2
    return is_hp, sorted(set(flags))

# ──────────────────────────────────────────────────────────────────────────────
# L1 — ROLE FIT
# ──────────────────────────────────────────────────────────────────────────────
def role_fit(c):
    title = (c.get("profile", {}).get("current_title", "") or "").lower()
    if RX_NON_TECHNICAL.search(title):   return 0.05, "non_technical"
    if RX_NON_TARGET_TECH.search(title): return 0.12, "non_target_tech"
    if RX_CORE_ML.search(title):         return 1.00, "core_ml"
    if RX_ADJACENT_ENG.search(title):    return 0.58, "adjacent_eng"
    return 0.30, "unknown_role"

# ──────────────────────────────────────────────────────────────────────────────
# L2 — DOMAIN EVIDENCE  (layered explicit / adjacent)
# ──────────────────────────────────────────────────────────────────────────────
def domain_evidence(c, text):
    ev = []
    retr   = len(RX_RETRIEVAL.findall(text))
    prod   = len(RX_PRODUCTION.findall(text))
    modern = len(RX_LLM_MODERN.findall(text))
    vdb    = len(RX_VECTOR_DB.findall(text))
    evalh  = len(RX_EVAL.findall(text))
    nlpir  = len(RX_NLP_IR.findall(text))
    adj    = len(RX_ADJACENT_ML.findall(text))
    cvrobo = len(RX_CV_SPEECH_ROBO.findall(text))

    retr_s   = 1 - math.exp(-retr / 2.0)
    modern_s = 1 - math.exp(-modern / 2.0)
    vdb_s    = min(vdb, 1)
    eval_s   = 1 - math.exp(-evalh / 1.5)
    nlpir_s  = 1 - math.exp(-nlpir / 3.0)
    explicit = (0.45*retr_s + 0.18*modern_s + 0.15*vdb_s + 0.12*eval_s + 0.10*nlpir_s)
    explicit_present = (retr + modern + vdb + evalh) > 0
    prod_s = 1 - math.exp(-prod / 2.0)
    explicit *= (0.7 + 0.3 * prod_s)
    adjacent = 1 - math.exp(-adj / 3.0)

    if explicit_present:
        score = float(np.clip(0.80*explicit + 0.20*adjacent, 0, 1))
        if retr:   ev.append("retrieval/ranking work")
        if vdb:    ev.append("vector/search infra")
        if modern: ev.append("modern LLM/embedding work")
        if evalh:  ev.append("ranking-evaluation experience")
        if prod and not ev: ev.append("production ML deployment")
    else:
        score = float(np.clip(adjacent, 0, 1) * 0.42)
        if adj: ev.append("adjacent ML/data work (plain-language)")

    cv_heavy = cvrobo >= 2 and (retr + nlpir) < cvrobo
    return score, ev, cv_heavy

# ──────────────────────────────────────────────────────────────────────────────
# L3 — FIT PILLARS
# ──────────────────────────────────────────────────────────────────────────────
def seniority_fit(c):
    yoe = float(c.get("profile", {}).get("years_of_experience") or 0)
    s = math.exp(-0.5 * ((yoe - JOB["ideal_yoe"]) / JOB["yoe_sigma"]) ** 2)
    if yoe < JOB["yoe_soft_min"]:
        s *= (max(yoe, 0) / JOB["yoe_soft_min"]) ** 2
    return float(s)

def product_vs_services(c):
    hist = c.get("career_history", [])
    if not hist: return 0.5
    def is_services(h):
        comp = (h.get("company", "") or "").lower()
        ind  = (h.get("industry", "") or "").lower()
        return any(f in comp for f in SERVICES_FIRMS) or "it services" in ind or "consult" in ind
    flags = [is_services(h) for h in hist]
    if not any(flags): return 1.0
    if all(flags):     return 0.25
    return 0.8

def skill_substance(c):
    sig    = c.get("redrob_signals", {})
    assess = sig.get("skill_assessment_scores", {}) or {}
    total, hits = 0.0, 0
    for s in c.get("skills", []):
        name = (s.get("name", "") or "").lower()
        if not (RX_RETRIEVAL.search(name) or RX_LLM_MODERN.search(name)
                or RX_PYTHON.search(name) or RX_NLP_IR.search(name)
                or RX_VECTOR_DB.search(name) or "ml" in name or "ai" in name
                or "data" in name or "statistic" in name):
            continue
        hits += 1
        pw  = PROF_W.get(s.get("proficiency"), 0.4)
        dur = min(int(s.get("duration_months") or 0) / 24.0, 1.0)
        end = min(math.log1p(int(s.get("endorsements") or 0)) / 4.0, 1.0)
        a   = assess.get(s.get("name", ""), None)
        asc = (a / 100.0) if isinstance(a, (int, float)) and a >= 0 else 0.5
        total += 0.4*pw + 0.2*dur + 0.15*end + 0.25*asc
    if hits == 0: return 0.0
    return float(np.clip(total / max(hits, 1) * min(hits / 4.0, 1.0), 0, 1))

def python_signal(c, text):
    for s in c.get("skills", []):
        if (s.get("name", "") or "").lower() == "python":
            return float(np.clip(0.5 + PROF_W.get(s.get("proficiency"), 0.4) * 0.5, 0, 1))
    return 0.6 if RX_PYTHON.search(text) else 0.25

def eval_frameworks(c, text):
    n = len(RX_EVAL.findall(text))
    return float(1 - math.exp(-n / 1.5)) if n else 0.15

def external_validation(c, text):
    gh = c.get("redrob_signals", {}).get("github_activity_score", -1)
    gh = float(gh) if gh is not None else -1.0
    base = 0.40 if gh < 0 else float(np.clip(0.45 + gh / 100.0 * 0.55, 0, 1))
    if RX_OSS.search(text): base = min(base + 0.15, 1.0)
    return base

def location_score(c):
    p       = c.get("profile", {})
    loc     = (p.get("location", "") or "").lower()
    country = (p.get("country", "") or "").lower()
    relo    = bool(c.get("redrob_signals", {}).get("willing_to_relocate", False))
    in_india = "india" in country or country == ""
    if any(city in loc for city in JOB["target_cities"]):           return 1.0
    if any(city in loc for city in JOB["good_india_cities"]):       return 0.90 if relo else 0.85
    if in_india:                                                     return 0.70 if relo else 0.45
    return 0.40 if relo else 0.20

def notice_score(c):
    nd = c.get("redrob_signals", {}).get("notice_period_days", 30)
    nd = float(nd) if nd is not None else 30.0
    return float(1.0 / (1.0 + math.exp(0.06 * (nd - JOB["notice_ideal_days"]))))

# ──────────────────────────────────────────────────────────────────────────────
# L4 — NEGATIVE PENALTIES
# ──────────────────────────────────────────────────────────────────────────────
def negative_penalty(c, text, domain_score, prod_serv, cv_heavy):
    reasons, ps = [], []
    if RX_RESEARCH.search(text) and not RX_PRODUCTION.search(text) and domain_score < 0.4:
        ps.append(0.55); reasons.append("research-leaning, little production signal")
    if prod_serv <= 0.25:
        ps.append(0.40); reasons.append("career entirely in IT-services/consulting")
    if cv_heavy:
        ps.append(0.35); reasons.append("CV/speech/robotics focus, thin NLP/IR")
    hist = c.get("career_history", [])
    noncur = [h for h in hist if not h.get("is_current")]
    short  = [h for h in noncur if int(h.get("duration_months") or 24) < 18]
    if len(noncur) >= 3 and len(short) >= 3:
        ps.append(0.22); reasons.append("frequent short stints (job-hopping pattern)")
    pen = 1.0
    for p in ps: pen *= (1 - p)
    return float(min(1 - pen, 0.85)), reasons

# ──────────────────────────────────────────────────────────────────────────────
# L5 — BEHAVIORAL MULTIPLIER
# ──────────────────────────────────────────────────────────────────────────────
def behavioral_multiplier(c, now):
    s   = c.get("redrob_signals", {})
    la  = _parse_date(s.get("last_active_date"))
    days = (now - la).days if la else 365
    recency = math.exp(-max(days, 0) / 120.0)
    resp    = float(s.get("recruiter_response_rate") or 0.0)
    icr     = float(s.get("interview_completion_rate") or 0.0)
    otw     = 1.0 if s.get("open_to_work_flag") else 0.55
    comp    = float(s.get("profile_completeness_score") or 0) / 100.0
    core = 0.40*recency + 0.25*resp + 0.15*icr + 0.10*otw + 0.10*comp
    return float(0.35 + 0.65 * np.clip(core, 0, 1)), {
        "days_inactive": days, "response_rate": resp,
        "open_to_work": bool(s.get("open_to_work_flag")),
        "behavior_core": float(np.clip(core, 0, 1)),
    }

def platform_quality_score(c):
    sig = c.get("redrob_signals", {}) or {}
    search = float(sig.get("search_appearance_30d") or 0)
    offer  = sig.get("offer_acceptance_rate")
    offer_f = float(offer) if offer is not None and float(offer) >= 0 else 0.5
    search_s = 1 - math.exp(-search / 300.0)
    return float(0.60 * search_s + 0.40 * offer_f)


def tier5_signal(c):
    """Platform engagement signal for head reordering in the Variant S cascade.
    t5 = 0.35*offer_s + 0.25*search_s + 0.20*saved_s + 0.20*assess_s
    """
    sig = c.get("redrob_signals", {}) or {}
    offer = sig.get("offer_acceptance_rate")
    offer_s  = float(offer) if offer is not None and float(offer) >= 0 else 0.0
    search_s = 1 - math.exp(-float(sig.get("search_appearance_30d") or 0) / 400.0)
    saved_s  = 1 - math.exp(-float(sig.get("saved_by_recruiters_30d") or 0) / 30.0)
    assess   = sig.get("skill_assessment_scores") or {}
    vals     = [v for v in assess.values() if isinstance(v, (int, float)) and v >= 0]
    assess_s = (sum(vals) / len(vals) / 100.0) if vals else 0.5
    return 0.35*offer_s + 0.25*search_s + 0.20*saved_s + 0.20*assess_s


# ──────────────────────────────────────────────────────────────────────────────
# SEMANTIC SIMILARITY — loaded from precomputed/ at rank time (no model inference)
# NOTE: the fallback path (precomputed/ absent) IS the canonical submission mode.
# WEIGHTS_FALLBACK produces the validated 0.7229 composite and is fully deterministic.
# Absence of precomputed/ is normal in the submission environment — not a degraded mode.
# ──────────────────────────────────────────────────────────────────────────────
_SEMANTIC_SCORES: dict = {}
_SEMANTIC_LOADED: bool = False

def _load_semantic_scores() -> None:
    global _SEMANTIC_SCORES, _SEMANTIC_LOADED
    if _SEMANTIC_LOADED:
        return
    emb_dir  = os.path.join(os.path.dirname(os.path.abspath(__file__)), "precomputed")
    emb_path = os.path.join(emb_dir, "candidate_embeddings.npy")
    jd_path  = os.path.join(emb_dir, "jd_embedding.npy")
    idx_path = os.path.join(emb_dir, "candidate_id_index.json")
    try:
        cand_emb = np.load(emb_path)   # [N, 768], L2-normalised
        jd_emb   = np.load(jd_path)    # [768],    L2-normalised
        sims     = (cand_emb @ jd_emb).astype(float)  # cosine sim in [0, 1]
        with open(idx_path, encoding="utf-8") as f:
            idx_map = json.load(f)
        for idx_str, cid in idx_map.items():
            _SEMANTIC_SCORES[cid] = float(np.clip(sims[int(idx_str)], 0.0, 1.0))
        print(f"  [semantic] {len(_SEMANTIC_SCORES):,} similarity scores loaded "
              f"(min={min(_SEMANTIC_SCORES.values()):.3f} "
              f"max={max(_SEMANTIC_SCORES.values()):.3f})", flush=True)
    except FileNotFoundError:
        global WEIGHTS, PILLAR_KEYS
        WEIGHTS = dict(WEIGHTS_FALLBACK)
        PILLAR_KEYS = list(WEIGHTS.keys())
        print("  [MODE] semantic embeddings absent → deterministic fallback weights "
              "(canonical submission mode)", flush=True)
    _SEMANTIC_LOADED = True

def semantic_similarity_score(c) -> float:
    cid = c.get("candidate_id", "")
    return _SEMANTIC_SCORES.get(cid, 0.5)

# ──────────────────────────────────────────────────────────────────────────────
# CORE SCORER — unchanged logic, used by both phases
# ──────────────────────────────────────────────────────────────────────────────
_NOW = date(2026, 6, 1)   # module-level so workers can access it

def score_candidate(c, now=None):
    if now is None: now = _NOW
    text = _candidate_text(c).lower()
    is_hp, hp = honeypot_flags(c)
    rmult, rlabel = role_fit(c)
    dscore, devid, cv_heavy = domain_evidence(c, text)
    prod_serv = product_vs_services(c)

    pillars = {
        "domain_evidence":     dscore,
        "skill_substance":     skill_substance(c),
        "seniority_fit":       seniority_fit(c),
        "product_vs_services": prod_serv,
        "external_validation": external_validation(c, text),
        "eval_frameworks":     eval_frameworks(c, text),
        "semantic_similarity": semantic_similarity_score(c),
        "python_signal":       python_signal(c, text),
        "location":            location_score(c),
        "notice":              notice_score(c),
        "platform_quality":    platform_quality_score(c),
    }
    base_fit = sum(WEIGHTS.get(k, 0) * v for k, v in pillars.items())
    if rlabel == "adjacent_eng":
        rmult = min(1.0, rmult + 0.45 * dscore)
    pen, neg_reasons = negative_penalty(c, text, dscore, prod_serv, cv_heavy)
    bmult, bdetail   = behavioral_multiplier(c, now)
    final = base_fit * rmult * (1 - pen) * bmult
    if is_hp:
        final = final * 0.01 - 0.2

    trace = {
        "final": float(final), "base_fit": float(base_fit), "role": rlabel,
        "role_mult": round(rmult, 3),
        "pillars": {k: round(v, 3) for k, v in pillars.items()},
        "domain_evidence_terms": devid, "penalty": round(pen, 3),
        "neg_reasons": neg_reasons, "behavior_mult": round(bmult, 3),
        "behavior": bdetail, "honeypot": is_hp, "honeypot_flags": hp,
    }
    return final, trace

# ──────────────────────────────────────────────────────────────────────────────
# MULTIPROCESSING WORKER  (must be top-level for Windows spawn compatibility)
# ──────────────────────────────────────────────────────────────────────────────
def _score_chunk(args):
    """Score a chunk of candidates. Top-level function for MP cross-platform."""
    chunk, now = args
    return [score_candidate(c, now) for c in chunk]

# ──────────────────────────────────────────────────────────────────────────────
# CONCEPT A — POPULATION CALIBRATION
# ──────────────────────────────────────────────────────────────────────────────
def _percentile_normalize(values):
    """Pure-numpy percentile rank. Returns array in [0, 1]."""
    n = len(values)
    if n <= 1: return np.zeros(n)
    order = np.argsort(values)
    ranks = np.empty(n, dtype=np.float32)
    ranks[order] = np.arange(n, dtype=np.float32)
    return ranks / (n - 1)

def population_calibrate(raw_results):
    """
    Concept A: replace the most arbitrary pillar scores with population
    percentile ranks. Scores now mean 'top X% of this pool' not 'absolute Y'.
    Only calibrates CALIBRATE_PILLARS (domain_evidence, skill_substance,
    eval_frameworks) — pillars with JD-relative meaning stay unchanged.
    Returns updated (score, trace) list with recomputed final scores.
    """
    # Collect raw pillar values across all candidates
    pillar_arrays = {k: np.array([tr["pillars"][k] for _, tr in raw_results],
                                 dtype=np.float32)
                     for k in CALIBRATE_PILLARS}

    # Compute percentile ranks for each calibratable pillar
    calibrated = {k: _percentile_normalize(pillar_arrays[k]) for k in CALIBRATE_PILLARS}

    # Recompute scores with calibrated pillars
    updated = []
    for i, (raw_score, tr) in enumerate(raw_results):
        new_pillars = dict(tr["pillars"])
        for k in CALIBRATE_PILLARS:
            new_pillars[k] = round(float(calibrated[k][i]), 4)
        new_base = sum(WEIGHTS.get(k, 0) * v for k, v in new_pillars.items())
        new_final = new_base * tr["role_mult"] * (1 - tr["penalty"]) * tr["behavior_mult"]
        if tr["honeypot"]:
            new_final = new_final * 0.01 - 0.2
        new_tr = dict(tr)
        new_tr["pillars"] = new_pillars
        new_tr["base_fit"] = round(float(new_base), 4)
        new_tr["final"]    = float(new_final)
        updated.append((float(new_final), new_tr))
    return updated

# ──────────────────────────────────────────────────────────────────────────────
# CONCEPT B — ISOLATION FOREST
# ──────────────────────────────────────────────────────────────────────────────
def _if_feature_vector(c):
    """Numeric feature vector for unsupervised anomaly detection."""
    p      = c.get("profile", {})
    s      = c.get("redrob_signals", {}) or {}
    hist   = c.get("career_history", [])
    skills = c.get("skills", [])
    yoe    = float(p.get("years_of_experience") or 0)
    career_months = sum(int(h.get("duration_months") or 0) for h in hist)
    expert_zero   = sum(1 for sk in skills
                        if sk.get("proficiency") in ("expert", "advanced")
                        and int(sk.get("duration_months") or 0) == 0)
    max_skill_dur = max((int(sk.get("duration_months") or 0) for sk in skills), default=0)
    gh            = float(s.get("github_activity_score") or 0); gh = max(gh, 0)
    completeness  = float(s.get("profile_completeness_score") or 0)
    endorsements  = float(s.get("endorsements_received") or 0)
    n_currents    = sum(1 for h in hist if h.get("is_current"))
    dur_ratio     = career_months / max(yoe * 12, 1)
    n_skills      = len(skills)
    return [yoe, career_months, expert_zero, max_skill_dur, n_skills,
            gh, completeness, endorsements, n_currents, dur_ratio]

def run_isolation_forest(candidates, calibrated_results):
    """
    Concept B: fit an Isolation Forest on ALL candidates to learn what
    'normal' looks like, then score anomalies. Returns an additional penalty
    array (0 = no penalty, up to 0.55 = strong anomaly signal).
    Falls back to zeros if sklearn is not installed.
    """
    n = len(candidates)
    penalties = np.zeros(n, dtype=np.float32)
    if not HAS_SKLEARN:
        return penalties

    print("  [B] Fitting Isolation Forest …", flush=True)
    X = np.array([_if_feature_vector(c) for c in candidates], dtype=np.float32)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    clf = IsolationForest(n_estimators=120, contamination=0.001,
                          random_state=42, n_jobs=-1)
    clf.fit(X_scaled)
    raw_scores = clf.score_samples(X_scaled)   # more negative = more anomalous

    # Normalise to [0, 1]: 0 = most normal, 1 = most anomalous
    lo, hi = raw_scores.min(), raw_scores.max()
    norm = 1.0 - (raw_scores - lo) / (hi - lo + 1e-8)

    # Only penalise candidates our rule-based gate DIDN'T already catch
    # (If rules flagged them, they're already crushed; IF covers the gaps)
    for i, (_, tr) in enumerate(calibrated_results):
        if not tr["honeypot"] and norm[i] > 0.85:
            # Anomalous but not rule-caught: apply graded penalty
            penalties[i] = float(np.clip((norm[i] - 0.85) / 0.15 * 0.55, 0, 0.55))
            tr["if_anomaly"] = round(float(norm[i]), 3)

    n_flagged = int((penalties > 0).sum())
    print(f"  [B] IF additional flags: {n_flagged} candidates", flush=True)
    return penalties

# ──────────────────────────────────────────────────────────────────────────────
# CONCEPT C — BOOTSTRAP RANK-CONFIDENCE
# ──────────────────────────────────────────────────────────────────────────────
def run_bootstrap(calibrated_results, top_n=100):
    """
    Concept C: perturb weights N_BOOTSTRAP times via matrix multiply.
    Returns confidence[i] = fraction of weight configs where candidate i
    appears in the top top_n. Near-instant after calibration (numpy BLAS).
    """
    n = len(calibrated_results)
    top_n = min(top_n, n - 1)   # guard: np.partition requires kth < n
    print(f"  [C] Bootstrap confidence ({N_BOOTSTRAP} samples) …", flush=True)

    # Build pillar matrix and adjustment vectors (both pre-calibrated)
    pillar_mat = np.array(
        [[tr["pillars"][k] for k in PILLAR_KEYS] for _, tr in calibrated_results],
        dtype=np.float32)                          # (N, 9)
    adjustments = np.array(
        [tr["role_mult"] * (1 - tr["penalty"]) * tr["behavior_mult"]
         for _, tr in calibrated_results],
        dtype=np.float32)                          # (N,)
    hp_mask = np.array([tr["honeypot"] for _, tr in calibrated_results])  # (N,)

    # Generate weight perturbations: log-normal around base weights
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    base_w = np.array([WEIGHTS[k] for k in PILLAR_KEYS], dtype=np.float32)
    log_p  = rng.normal(0, 0.35, (N_BOOTSTRAP, len(base_w))).astype(np.float32)
    w_samp = base_w * np.exp(log_p)
    w_samp *= (base_w.sum() / w_samp.sum(axis=1, keepdims=True))  # renormalise

    # Vectorised scoring: (N, 9) @ (9, B) = (N, B)
    bs_scores = pillar_mat @ w_samp.T                  # (N, B)
    bs_scores *= adjustments[:, None]                  # apply role/penalty/behav
    bs_scores[hp_mask] = -1.0                          # crush honeypots

    # For each bootstrap sample, find the score of the top_n-th candidate
    # np.partition is O(N) per column — much faster than full sort
    neg_part   = np.partition(-bs_scores, top_n, axis=0)  # partial sort
    thresholds = -neg_part[top_n - 1, :]                  # (B,) per-sample cutoff

    # Confidence = fraction of samples where this candidate beats the cutoff
    # We only need it for the candidates we'll actually report (top ~200)
    # but compute for all N for correctness then slice
    confidence = np.zeros(n, dtype=np.float32)
    # Process in batches to stay memory-light (avoid 100K × 500 bool matrix)
    batch = 500
    for start in range(0, n, batch):
        end = min(start + batch, n)
        chunk = bs_scores[start:end, :]                   # (batch, B)
        conf  = (chunk >= thresholds[None, :]).mean(axis=1)
        confidence[start:end] = conf

    return confidence

# ──────────────────────────────────────────────────────────────────────────────
# TIER PREDICTION
# ──────────────────────────────────────────────────────────────────────────────
def predict_tier(trace):
    if trace["honeypot"] or trace["role"] in ("non_technical", "non_target_tech"):
        return 0
    f = trace["final"]
    if f >= 0.749: return 5
    if f >= 0.606: return 4
    if f >= 0.462: return 3
    if f >= 0.426: return 2
    if f >= 0.390: return 1
    return 0

def make_reasoning(c, trace, tier, rank_stability=None, rank=0):
    title = c.get("profile", {}).get("current_title", "professional")

    # Edge-case guards — these candidates won't normally reach top-100
    if trace["honeypot"]:
        return (f"{title} profile contains internal inconsistencies "
                f"({', '.join(trace['honeypot_flags'][:2])}); "
                f"flagged as non-credible and ranked at the bottom.")
    if trace.get("if_anomaly", 0) > 0.85 and not trace["honeypot"]:
        return (f"{title}; statistical anomaly signals detected "
                f"(score {trace['if_anomaly']:.2f}); "
                f"recommend manual profile verification.")
    if trace["role"] == "non_technical":
        return (f"Current role is {title}, a non-engineering function; "
                f"AI skills are listed but the role does not match the JD "
                f"regardless of keywords.")

    try:
        from reasoning import make_rich_reasoning
        text = make_rich_reasoning(c, trace, rank)
    except ImportError:
        print("  [warn] reasoning.py not found — using inline fallback", flush=True)
        concerns = trace.get("neg_reasons", [])
        top_concern = f" Note: {concerns[0]}." if concerns else ""
        text = f"{title} (tier {tier}).{top_concern}"

    # Rank-stability annotation: sensitivity to weight perturbation, NOT a ranking input
    if rank_stability is not None:
        if rank_stability >= 0.85:
            text += f" Rank stability: {rank_stability:.0%} of weight perturbations agree."
        elif rank_stability >= 0.50:
            text += f" Rank stability: {rank_stability:.0%} of weight perturbations agree; verify against JD."
        else:
            text += f" Rank stability: {rank_stability:.0%} of weight perturbations agree; manual review recommended."

    return text[:350]

# ──────────────────────────────────────────────────────────────────────────────
# MAIN ORCHESTRATOR
# ──────────────────────────────────────────────────────────────────────────────
def rank_all(candidates, top_n=100):
    n = len(candidates)
    if n == 0:
        return [], []

    now = date(2026, 6, 1)  # fixed reference date — intentional for deterministic recency scoring

    # ── Pre-phase: load semantic similarity scores from precomputed embeddings ─
    _load_semantic_scores()

    # ── Phase 1: multiprocessing feature extraction ───────────────────────────
    ncores = cpu_count()
    chunk_size = max(1, n // ncores)
    chunks = [candidates[i:i+chunk_size] for i in range(0, n, chunk_size)]
    args   = [(chunk, now) for chunk in chunks]

    if ncores > 1 and n > 200:
        with Pool(processes=ncores) as pool:
            chunk_results = pool.map(_score_chunk, args)
    else:
        chunk_results = [_score_chunk(a) for a in args]

    raw_results = [item for sublist in chunk_results for item in sublist]
    # raw_results: [(score, trace), ...]

    # ── Phase 2: population calibration (Concept A) ───────────────────────────
    print("  [A] Population calibration …", flush=True)
    calibrated = population_calibrate(raw_results)
    # calibrated: [(score, trace), ...]

    # ── Phase 3: Isolation Forest (Concept B) ─────────────────────────────────
    if_penalties = run_isolation_forest(candidates, calibrated)

    # Apply IF penalties to final_order scores
    for i, (score, tr) in enumerate(calibrated):
        pen = float(if_penalties[i])
        if pen > 0:
            new_score = score * (1 - pen)
            tr["final"] = new_score
            calibrated[i] = (new_score, tr)

    # Compute final_select (behavior_mult = 1.0); honeypot crush and IF apply
    final_selects = []
    for i, (score, tr) in enumerate(calibrated):
        fs = tr["base_fit"] * tr["role_mult"] * (1 - tr["penalty"])
        if tr["honeypot"]:
            fs = fs * 0.01 - 0.2
        pen_i = float(if_penalties[i])
        if pen_i > 0:
            fs *= (1 - pen_i)
        final_selects.append(float(fs))

    # ── Phase 4: bootstrap confidence (Concept C) ─────────────────────────────
    confidence = run_bootstrap(calibrated, top_n=top_n)

    # ── Phase 5: Variant S cascade — two-stage selection/ordering decomposition ──
    scored = [(candidates[i], calibrated[i][0], calibrated[i][1], float(confidence[i]), final_selects[i])
              for i in range(n)]

    # Stage 1: pool = top-100 by final_select (selects quality candidates, ignores availability noise)
    pool = sorted(scored, key=lambda x: -x[4])[:top_n]

    # Head: top-15 of pool by final_order, reordered by final_order*(0.85+0.15*t5)
    pool_by_order = sorted(pool, key=_cascade_sort_key)
    head = sorted(pool_by_order[:15],
                  key=lambda it: -(it[1] * (0.85 + 0.15 * tier5_signal(it[0]))))

    # Tail: remaining 85 by calibrated domain_evidence desc;
    # tail tie-runs on domain_evidence are intentional — tiebreak: behavior_core desc, candidate_id asc
    tail = sorted(pool_by_order[15:],
                  key=lambda it: (-it[2]["pillars"]["domain_evidence"],
                                  -it[2]["behavior"].get("behavior_core", 0),
                                  it[0].get("candidate_id", "")))

    rows = _make_cascade_rows(head + tail, top_n)
    return rows, scored

# ──────────────────────────────────────────────────────────────────────────────
# OUTPUT
# ──────────────────────────────────────────────────────────────────────────────
def write_csv(rows, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
        w.writerow(["candidate_id", "rank", "score", "reasoning"])
        for r in rows:
            w.writerow([r["candidate_id"], r["rank"], r["score"], r["reasoning"]])

# ──────────────────────────────────────────────────────────────────────────────
# FEATURE EXPORT  (for LambdaMART training via train_ltr.py)
# ──────────────────────────────────────────────────────────────────────────────
def _export_pillar_features(scored, path):
    """
    Write a CSV of every candidate's calibrated pillar scores.
    Column names match train_ltr.py's FEATURE_COLS exactly so no renaming
    is needed downstream.
    scored: [(candidate, final_score, trace, confidence), ...]  from rank_all()
    """
    headers = [
        "candidate_id",
        "domain_evidence_cal", "skill_substance_cal",
        "seniority_fit", "product_vs_services", "external_validation",
        "eval_frameworks_cal", "semantic_similarity", "python_signal",
        "location_score", "notice_score", "platform_quality",
        "behavioral_mult", "role_mult", "neg_penalty",
        "is_honeypot", "base_fit", "final_score",
    ]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
        writer.writeheader()
        for c, final, trace, conf, *_ in scored:
            pil = trace["pillars"]
            writer.writerow({
                "candidate_id":        c.get("candidate_id", ""),
                "domain_evidence_cal": pil.get("domain_evidence", 0),
                "skill_substance_cal": pil.get("skill_substance", 0),
                "seniority_fit":       pil.get("seniority_fit", 0),
                "product_vs_services": pil.get("product_vs_services", 0),
                "external_validation": pil.get("external_validation", 0),
                "eval_frameworks_cal": pil.get("eval_frameworks", 0),
                "semantic_similarity": pil.get("semantic_similarity", 0),
                "python_signal":       pil.get("python_signal", 0),
                "location_score":      pil.get("location", 0),
                "notice_score":        pil.get("notice", 0),
                "platform_quality":    pil.get("platform_quality", 0),
                "behavioral_mult":     trace.get("behavior_mult", 0),
                "role_mult":           trace.get("role_mult", 0),
                "neg_penalty":         trace.get("penalty", 0),
                "is_honeypot":         int(trace.get("honeypot", False)),
                "base_fit":            trace.get("base_fit", 0),
                "final_score":         float(final),
            })
    print(f"  [D] Pillar features → {path}  ({len(scored):,} rows)", flush=True)


# ──────────────────────────────────────────────────────────────────────────────
# CASCADE VARIANT BUILDER  (two-stage: select by final_select, rank by final_order)
# ──────────────────────────────────────────────────────────────────────────────
def _cascade_sort_key(item):
    return (-item[1], -item[2]["behavior"].get("behavior_core", 0), item[0].get("candidate_id", ""))


def _make_cascade_rows(pool, top_n=100):
    rows, prev = [], float("inf")
    for rank, (c, final, trace, conf, _fs) in enumerate(pool[:top_n], start=1):
        tier  = predict_tier(trace)
        score = round(min(final, prev), 6)
        prev  = score
        rows.append({
            "candidate_id":    c.get("candidate_id", ""),
            "rank":            rank,
            "score":           score,
            "reasoning":       make_reasoning(c, trace, tier, rank_stability=conf, rank=rank),
            "_tier":           tier,
            "_trace":          trace,
            "_rank_stability": conf,
        })
    return rows


def main():
    ap = argparse.ArgumentParser(
        description="Redrob Ranking Engine — glass-box, deterministic, CPU-only candidate ranker."
    )
    ap.add_argument("--candidates",   required=True,
                    help="Candidate pool JSONL (supports .jsonl.gz compression)")
    ap.add_argument("--out",          default="submission.csv",
                    help="Output submission CSV (default: submission.csv)")
    ap.add_argument("--top",          type=int, default=100,
                    help="Number of candidates to rank (default: 100)")
    ap.add_argument("--dump-pillars", default=None, dest="dump_pillars",
                    help="Write pillar feature CSV to this path (for LambdaMART training)")
    ap.add_argument("--experiments",  action="store_true",
                    help="Run all A-W variant experiments; imports experiments.py")
    ap.add_argument("--labels",       default="labels_filled 500.csv",
                    help="Labels CSV used by --experiments (default: 'labels_filled 500.csv')")
    args = ap.parse_args()

    t0 = datetime.now()
    print(f"Loading {args.candidates} …")
    cands = load_candidates(args.candidates)
    print(f"  {len(cands):,} candidates loaded")

    rows, scored = rank_all(cands, top_n=args.top)
    write_csv(rows, args.out)

    if args.dump_pillars:
        _export_pillar_features(scored, args.dump_pillars)

    hp = sum(1 for r in rows if r["_trace"]["honeypot"])
    nt = sum(1 for r in rows if r["_trace"]["role"] in ("non_technical", "non_target_tech"))
    dt = (datetime.now() - t0).total_seconds()
    print(f"\n  wrote {len(rows)} rows -> {args.out}")
    print(f"  honeypots in top-{args.top}: {hp} ({hp/max(len(rows),1):.1%})  "
          f"| non-target roles: {nt}")
    print(f"  runtime: {dt:.1f}s")
    print(f"\n  Top 10:")
    for r in rows[:10]:
        conf_str = f"stab={r['_rank_stability']:.0%}"
        print(f"    {r['rank']:>3}. {r['candidate_id']}  "
              f"s={r['score']:.4f} t{r['_tier']} {conf_str}  "
              f"{r['reasoning'][:80]}")

    if args.experiments:
        import experiments
        experiments.run_all_experiments(scored, labels_path=args.labels)


if __name__ == "__main__":
    main()


## 3. Load the demo candidate set
A curated 12-candidate sample showcasing every differentiator. (You can also upload your own file in the next cell.)

In [ ]:
%%writefile demo_candidates.json
[
  {
    "candidate_id": "CAND_0000001",
    "profile": {
      "anonymized_name": "A. Sharma",
      "headline": "Senior ML Engineer | Search & Recommendation Systems",
      "summary": "Built and shipped large-scale retrieval and ranking systems serving millions of users. Deep experience with embeddings, vector search, and offline/online evaluation.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 7,
      "current_title": "Senior Machine Learning Engineer",
      "current_company": "Flipkart",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Flipkart",
        "title": "Senior ML Engineer",
        "start_date": "2023-01-01",
        "end_date": null,
        "duration_months": 36,
        "is_current": true,
        "industry": "E-commerce",
        "company_size": "1001-5000",
        "description": "Led the product recommendation engine using embeddings and approximate nearest neighbour retrieval with FAISS. Improved NDCG by 18% via learning-to-rank. Ran A/B tests and offline NDCG/MRR evaluation. Deployed to production serving 10M users at low latency."
      },
      {
        "company": "Myntra",
        "title": "ML Engineer",
        "start_date": "2019-01-01",
        "end_date": "2022-12-31",
        "duration_months": 48,
        "is_current": false,
        "industry": "E-commerce",
        "company_size": "1001-5000",
        "description": "Built semantic search with sentence-transformers and Elasticsearch hybrid retrieval. Owned embedding refresh pipeline and retrieval-quality regression monitoring in production."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Python",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 84
      },
      {
        "name": "PyTorch",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 60
      },
      {
        "name": "Retrieval",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 72
      },
      {
        "name": "Embeddings",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 60
      },
      {
        "name": "FAISS",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 48
      },
      {
        "name": "Elasticsearch",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 54
      },
      {
        "name": "Learning to Rank",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 40
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.9,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {
        "Python": 92,
        "Retrieval": 88
      },
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 85,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000002",
    "profile": {
      "anonymized_name": "B. Reddy",
      "headline": "AI Engineer | RAG, LLMs, Vector Search",
      "summary": "Applied AI engineer focused on retrieval-augmented generation and hybrid search. Shipped LLM-powered features to real users.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 6,
      "current_title": "AI Engineer",
      "current_company": "Swiggy",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Swiggy",
        "title": "AI Engineer",
        "start_date": "2022-11-01",
        "end_date": null,
        "duration_months": 40,
        "is_current": true,
        "industry": "FoodTech",
        "company_size": "1001-5000",
        "description": "Built a RAG pipeline with Qdrant vector database and fine-tuned embedding models (BGE) for semantic product search. Designed the ranking and reranking layer. Set up offline evaluation with NDCG and MRR plus online A/B testing."
      },
      {
        "company": "Zomato",
        "title": "ML Engineer",
        "start_date": "2020-03-01",
        "end_date": "2022-10-31",
        "duration_months": 32,
        "is_current": false,
        "industry": "FoodTech",
        "company_size": "1001-5000",
        "description": "Developed personalization and ranking models. Production deployment with latency monitoring."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Python",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 72
      },
      {
        "name": "RAG",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 36
      },
      {
        "name": "LLM",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 36
      },
      {
        "name": "Qdrant",
        "proficiency": "intermediate",
        "endorsements": 30,
        "duration_months": 30
      },
      {
        "name": "Embeddings",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 40
      },
      {
        "name": "NDCG",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 36
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.85,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {
        "Python": 90
      },
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 78,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true,
      "location": "Noida, Uttar Pradesh"
    }
  },
  {
    "candidate_id": "CAND_0000003",
    "profile": {
      "anonymized_name": "C. Iyer",
      "headline": "Backend Engineer | Data-Intensive Systems",
      "summary": "Backend engineer who built the system that suggests products to users and the pipelines feeding our data science team. Comfortable with model training and experimentation.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 7,
      "current_title": "Backend Engineer",
      "current_company": "Razorpay",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Razorpay",
        "title": "Backend Engineer",
        "start_date": "2022-09-01",
        "end_date": null,
        "duration_months": 44,
        "is_current": true,
        "industry": "Fintech",
        "company_size": "1001-5000",
        "description": "Built the recommendation system that suggests relevant products to users at checkout. Owned the feature pipelines and experimentation framework feeding the data science team. Shipped model training and serving infrastructure to production."
      },
      {
        "company": "Paytm",
        "title": "Software Engineer",
        "start_date": "2019-04-01",
        "end_date": "2022-08-31",
        "duration_months": 40,
        "is_current": false,
        "industry": "Fintech",
        "company_size": "1001-5000",
        "description": "Built data pipelines and ETL with Spark and Airflow. Worked closely with the data science team on churn prediction."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Python",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 72
      },
      {
        "name": "Spark",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 48
      },
      {
        "name": "Airflow",
        "proficiency": "intermediate",
        "endorsements": 30,
        "duration_months": 40
      },
      {
        "name": "Data Pipeline",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 60
      },
      {
        "name": "Machine Learning",
        "proficiency": "intermediate",
        "endorsements": 30,
        "duration_months": 36
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.7,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {},
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 55,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true,
      "loc": "Bangalore, Karnataka"
    }
  },
  {
    "candidate_id": "CAND_0000004",
    "profile": {
      "anonymized_name": "D. Kapoor",
      "headline": "HR Manager | RAG, LLM, Embeddings, Retrieval, NLP",
      "summary": "RAG retrieval ranking embeddings vector search NDCG fine-tuning LLM transformers deep learning machine learning.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 8,
      "current_title": "HR Manager",
      "current_company": "TechCorp",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "TechCorp",
        "title": "HR Manager",
        "start_date": "2018-01-01",
        "end_date": null,
        "duration_months": 96,
        "is_current": true,
        "industry": "HR",
        "company_size": "1001-5000",
        "description": "RAG retrieval ranking embeddings vector search NDCG fine-tuning LLM. Managed hiring and employee relations."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "RAG",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 40
      },
      {
        "name": "LLM",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 40
      },
      {
        "name": "Embeddings",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 40
      },
      {
        "name": "Retrieval",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 40
      },
      {
        "name": "NLP",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 40
      },
      {
        "name": "Ranking",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 40
      },
      {
        "name": "Deep Learning",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 40
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.9,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {},
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 80,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000005",
    "profile": {
      "anonymized_name": "E. Singh",
      "headline": "Senior ML Engineer | Retrieval, Ranking, RAG",
      "summary": "Expert across the full modern ML stack with deep retrieval and ranking experience.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 9,
      "current_title": "Senior Machine Learning Engineer",
      "current_company": "BigAI",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "BigAI",
        "title": "Senior ML Engineer",
        "start_date": "2023-01-01",
        "end_date": null,
        "duration_months": 36,
        "is_current": true,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Retrieval ranking RAG embeddings production systems."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "RAG",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 0
      },
      {
        "name": "Retrieval",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 0
      },
      {
        "name": "Ranking",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 0
      },
      {
        "name": "LLM",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 0
      },
      {
        "name": "NLP",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 0
      },
      {
        "name": "Embeddings",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 0
      },
      {
        "name": "PyTorch",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 0
      },
      {
        "name": "Search",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 0
      },
      {
        "name": "Vector DB",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 0
      },
      {
        "name": "MLOps",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 0
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 99,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.9,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {},
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 90,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000006",
    "profile": {
      "anonymized_name": "F. Nair",
      "headline": "Technical Lead | Enterprise Solutions",
      "summary": "Delivered enterprise software projects across multiple domains.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 8,
      "current_title": "Machine Learning Engineer",
      "current_company": "Infosys",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Infosys",
        "title": "ML Engineer",
        "start_date": "2022-01-01",
        "end_date": null,
        "duration_months": 50,
        "is_current": true,
        "industry": "IT Services",
        "company_size": "1001-5000",
        "description": "Built ML classification models for client projects. Data pipelines and analytics."
      },
      {
        "company": "TCS",
        "title": "Software Engineer",
        "start_date": "2018-01-01",
        "end_date": "2021-12-31",
        "duration_months": 46,
        "is_current": false,
        "industry": "IT Services",
        "company_size": "1001-5000",
        "description": "Developed enterprise applications for banking clients."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Python",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 60
      },
      {
        "name": "Machine Learning",
        "proficiency": "intermediate",
        "endorsements": 30,
        "duration_months": 48
      },
      {
        "name": "Classification",
        "proficiency": "intermediate",
        "endorsements": 30,
        "duration_months": 40
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.6,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {},
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": -1,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000007",
    "profile": {
      "anonymized_name": "G. Bose",
      "headline": "Senior ML Engineer | Search & Ranking",
      "summary": "Built search and ranking systems with embeddings and vector retrieval in production.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 7,
      "current_title": "Senior Machine Learning Engineer",
      "current_company": "Meesho",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Meesho",
        "title": "Senior ML Engineer",
        "start_date": "2022-11-01",
        "end_date": null,
        "duration_months": 40,
        "is_current": true,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Built semantic search with embeddings and FAISS retrieval. Ranking models with NDCG evaluation. Production deployment."
      },
      {
        "company": "Ola",
        "title": "ML Engineer",
        "start_date": "2019-10-01",
        "end_date": "2022-10-31",
        "duration_months": 36,
        "is_current": false,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Recommendation and ranking systems at scale."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Python",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 72
      },
      {
        "name": "Retrieval",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 48
      },
      {
        "name": "Embeddings",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 40
      },
      {
        "name": "FAISS",
        "proficiency": "intermediate",
        "endorsements": 30,
        "duration_months": 36
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2025-11-01",
      "open_to_work_flag": false,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.05,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {},
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 72,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.2,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000008",
    "profile": {
      "anonymized_name": "H. Menon",
      "headline": "Computer Vision Engineer | Deep Learning",
      "summary": "Specialist in image classification, object detection, and segmentation for autonomous systems.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 7,
      "current_title": "Computer Vision Engineer",
      "current_company": "Ather",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Ather",
        "title": "Computer Vision Engineer",
        "start_date": "2022-01-01",
        "end_date": null,
        "duration_months": 48,
        "is_current": true,
        "industry": "Automotive",
        "company_size": "1001-5000",
        "description": "Built object detection and image segmentation models for autonomous vehicles. Point cloud processing and SLAM. Production deployment on edge devices."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Python",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 72
      },
      {
        "name": "PyTorch",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 60
      },
      {
        "name": "Computer Vision",
        "proficiency": "expert",
        "endorsements": 30,
        "duration_months": 60
      },
      {
        "name": "Object Detection",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 48
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.7,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {},
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 68,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000009",
    "profile": {
      "anonymized_name": "I. Das",
      "headline": "Software Engineer | Full Stack",
      "summary": "Full-stack engineer with backend focus.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 5,
      "current_title": "Software Engineer",
      "current_company": "Zerodha",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Zerodha",
        "title": "Software Engineer",
        "start_date": "2022-09-01",
        "end_date": null,
        "duration_months": 40,
        "is_current": true,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Built backend APIs and microservices. Some exposure to analytics and data pipelines."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Python",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 48
      },
      {
        "name": "Data Pipeline",
        "proficiency": "intermediate",
        "endorsements": 30,
        "duration_months": 24
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.8,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {},
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 50,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true,
      "loc": "Chennai, Tamil Nadu"
    }
  },
  {
    "candidate_id": "CAND_0000010",
    "profile": {
      "anonymized_name": "J. Roy",
      "headline": "Data Engineer | Pipelines",
      "summary": "Data engineer building large-scale ETL.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 6,
      "current_title": "Data Engineer",
      "current_company": "PhonePe",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "PhonePe",
        "title": "Data Engineer",
        "start_date": "2022-06-01",
        "end_date": null,
        "duration_months": 44,
        "is_current": true,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Built data warehouse and ETL pipelines with Spark and Airflow. Feature engineering for ML team. Some model training experience."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Python",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 60
      },
      {
        "name": "Spark",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 48
      },
      {
        "name": "ETL",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 48
      },
      {
        "name": "Machine Learning",
        "proficiency": "beginner",
        "endorsements": 30,
        "duration_months": 18
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.8,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {},
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 58,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true
    }
  },
  {
    "candidate_id": "CAND_0000011",
    "profile": {
      "anonymized_name": "K. Gupta",
      "headline": "Frontend Engineer | React",
      "summary": "Frontend specialist.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 6,
      "current_title": "Frontend Engineer",
      "current_company": "CRED",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "CRED",
        "title": "Frontend Engineer",
        "start_date": "2022-01-01",
        "end_date": null,
        "duration_months": 48,
        "is_current": true,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Built React applications and design systems."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Python",
        "proficiency": "beginner",
        "endorsements": 30,
        "duration_months": 12
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.8,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {},
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 40,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true,
      "loc": "Mumbai, Maharashtra"
    }
  },
  {
    "candidate_id": "CAND_0000012",
    "profile": {
      "anonymized_name": "L. Verma",
      "headline": "NLP Engineer | Text Analytics",
      "summary": "NLP engineer with retrieval and ranking experience.",
      "location": "Pune, Maharashtra",
      "country": "India",
      "years_of_experience": 5,
      "current_title": "NLP Engineer",
      "current_company": "Glance",
      "current_company_size": "1001-5000",
      "current_industry": "Software"
    },
    "career_history": [
      {
        "company": "Glance",
        "title": "NLP Engineer",
        "start_date": "2023-01-01",
        "end_date": null,
        "duration_months": 36,
        "is_current": true,
        "industry": "Software",
        "company_size": "1001-5000",
        "description": "Built text classification and named entity recognition. Search relevance and ranking with embeddings. Offline evaluation with precision and recall."
      }
    ],
    "education": [
      {
        "institution": "IIT",
        "degree": "B.Tech",
        "field_of_study": "Computer Science",
        "start_year": 2012,
        "end_year": 2016,
        "grade": "8.5",
        "tier": "tier_1"
      }
    ],
    "skills": [
      {
        "name": "Python",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 48
      },
      {
        "name": "NLP",
        "proficiency": "advanced",
        "endorsements": 30,
        "duration_months": 40
      },
      {
        "name": "Embeddings",
        "proficiency": "intermediate",
        "endorsements": 30,
        "duration_months": 30
      },
      {
        "name": "Search",
        "proficiency": "intermediate",
        "endorsements": 30,
        "duration_months": 30
      }
    ],
    "redrob_signals": {
      "profile_completeness_score": 85,
      "signup_date": "2024-01-15",
      "last_active_date": "2026-05-28",
      "open_to_work_flag": true,
      "profile_views_received_30d": 40,
      "applications_submitted_30d": 5,
      "recruiter_response_rate": 0.8,
      "avg_response_time_hours": 6,
      "skill_assessment_scores": {},
      "connection_count": 500,
      "endorsements_received": 60,
      "notice_period_days": 30,
      "expected_salary_range_inr_lpa": {
        "min": 30,
        "max": 50
      },
      "preferred_work_mode": "hybrid",
      "willing_to_relocate": true,
      "github_activity_score": 62,
      "search_appearance_30d": 50,
      "saved_by_recruiters_30d": 8,
      "interview_completion_rate": 0.9,
      "offer_acceptance_rate": 0.5,
      "verified_email": true,
      "verified_phone": true,
      "linkedin_connected": true,
      "loc": "Hyderabad, Telangana"
    }
  }
]

## 4. (Optional) Upload your own candidates file
Run this cell to upload a `.json` or `.jsonl` file (≤100 candidates) in the released schema.
**To just use the built-in demo set, click _Cancel_ on the upload dialog (or skip this cell).**

In [ ]:
CANDIDATES_FILE = "demo_candidates.json"
try:
    from google.colab import files
    print("Upload a candidates .json/.jsonl file — or click Cancel to use the demo set:")
    uploaded = files.upload()
    if uploaded:
        CANDIDATES_FILE = list(uploaded.keys())[0]
except Exception:
    pass
print("Using candidates file:", CANDIDATES_FILE)

## 5. Run the ranker
Produces `submission.csv` and displays the ranked table with score, tier, and bootstrap confidence.

In [ ]:
import importlib, rank as R
importlib.reload(R)
import pandas as pd

cands = R.load_candidates(CANDIDATES_FILE)
top_n = min(100, len(cands))
rows, _ = R.rank_all(cands, top_n=top_n)
R.write_csv(rows, "submission.csv")

pd.set_option("display.max_colwidth", None)
df = pd.DataFrame([{
    "rank": r["rank"],
    "candidate_id": r["candidate_id"],
    "score": r["score"],
    "tier": r["_tier"],
    "confidence": f"{r['_confidence']:.0%}",
    "reasoning": r["reasoning"],
} for r in rows])
print(f"\nRanked {len(cands)} candidates -> top {top_n} written to submission.csv\n")
df

In [ ]:
# ── HTML report ──────────────────────────────────────────────────────────────
from rank import generate_html_report

generate_html_report(rows, cands, "report.html")

try:
    import google.colab  # noqa: F401
    from IPython.display import HTML, display
    display(HTML(open("report.html").read()))
except ImportError:
    pass  # not Colab — generate_html_report already opened via open/xdg-open/start


## 6. Download the ranked CSV

In [ ]:
try:
    from google.colab import files
    files.download("submission.csv")
except Exception:
    print("submission.csv is in the working directory.")

## What you're seeing (differentiators)

On the built-in demo set, note how the engine behaves — this is the whole point of the design:

| Behavior | Which candidate | Why it matters |
|---|---|---|
| **Explicit fits rise to the top (tier 5)** | ML/AI/NLP engineers with retrieval, RAG, ranking | Reads *production evidence*, not keywords |
| **Plain-language fit is caught** | the "Backend Engineer" who *"built the system that suggests products"* | Catches Tier-5s who never say "RAG" or "Pinecone" |
| **Keyword-stuffer is floored (tier 0)** | the "HR Manager" loaded with RAG/LLM/embeddings | Role-fit gate beats keyword matching |
| **Honeypot is floored (negative score)** | the "ML Engineer" with 10 expert skills at 0 months used | Two independent defenses: consistency rules + Isolation Forest |
| **Unresponsive candidate is down-weighted** | the strong-on-paper engineer inactive 6 months, 5% response | Behavioral signals as a multiplier (per the JD) |
| **Consulting-only career is penalized** | the ML engineer entirely at IT-services firms | Matches the JD's explicit do-NOT-want list |

> On the demo set of 12, the floored candidates still appear (ranks 9–12) simply because there are only 12 rows to show. On the full 100K pool they are buried near rank ~40,000+. The honeypot-rate warning printed during the run reflects this small-pool artifact, not the real submission.

**The reasoning column is generated from the same numbers that produced the rank** — it cannot praise a buried candidate or cite a skill they don't have. That is our edge in the manual-review and defend-your-work stages.